# BLSEconomicSurveys - Data Processing

Fetches BLS economic survey data (CPI, CES, PPI, JOLTS) via the BLS API v2
and scrapes BLS release schedule pages for accurate publication dates.

**Workflow:**
1. Scrape BLS release schedule pages (2000-2026) for exact release dates
2. Fetch series data from BLS API v2 (20-year windows)
3. Join data with release dates to set accurate EndTime
4. Write wide-format CSVs: `time,endtime,series1,series2,...`

**Processing Times** (fill in after running):
- Full dataset: `<TIME>`
- One day update: `<TIME>`

In [ ]:
import json
import os
import re
import time
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
import requests

## Configuration

In [ ]:
config_path = Path("config.json")
with open(config_path) as f:
    config = json.load(f)

API_KEY = config.get("bls-api-key", "")
TEMP_OUTPUT_DIR = config.get("temp-output-directory", "C:/temp-output-directory")
LEAN_DATA_FOLDER = config.get("data-folder", "C:/Users/derek/claude/Lean/Data/")

deployment_date_str = os.environ.get("QC_DATAFLEET_DEPLOYMENT_DATE", "")
INCREMENTAL_MODE = bool(deployment_date_str)

RELATIVE_DATA_PATH = os.path.join("alternative", "bls", "economicsurveys")

BLS_API_URL = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

print(f"Incremental mode: {INCREMENTAL_MODE}")
if INCREMENTAL_MODE:
    print(f"Deployment date : {deployment_date_str}")
print(f"Data folder     : {LEAN_DATA_FOLDER}")
print(f"Temp output     : {TEMP_OUTPUT_DIR}")

## Survey Definitions

In [ ]:
SURVEYS = {
    "cpi": {
        "schedule_name": "Consumer Price Index",
        "release_time": "08:30",
        "series": {
            "CUUR0000SA0": "AllItems",
            "CUUR0000SA0L1E": "CoreCpi",
            "CUUR0000SAF1": "Food",
            "CUUR0000SAF11": "FoodAtHome",
            "CUUR0000SEFV": "FoodAwayFromHome",
            "CUUR0000SA0E": "Energy",
            "CUUR0000SAH1": "Shelter",
            "CUUR0000SEHA": "RentOfPrimaryResidence",
            "CUUR0000SETB01": "Gasoline",
            "CUUR0000SAM": "MedicalCare",
            "CUUR0000SAA": "Apparel",
            "CUUR0000SAE": "EducationAndCommunication",
            "CUUR0000SETA01": "NewVehicles",
            "CUUR0000SETA02": "UsedCarsAndTrucks",
            "CUUR0000SEEB01": "CollegeTuitionAndFees",
        },
    },
    "ces": {
        "schedule_name": "Employment Situation",
        "release_time": "08:30",
        "series": {
            "CEU0000000001": "TotalNonfarm",
            "CEU0500000001": "TotalPrivate",
            "CEU0500000003": "AverageHourlyEarnings",
            "CEU0500000002": "AverageWeeklyHours",
            "CEU0500000011": "AverageWeeklyEarnings",
            "CEU0500000008": "ProductionHourlyEarnings",
            "CEU0500000006": "ProductionEmployees",
            "CEU3000000001": "Manufacturing",
            "CEU0600000001": "GoodsProducing",
            "CEU0800000001": "PrivateServiceProviding",
            "CEU2000000001": "Construction",
            "CEU4200000001": "RetailTrade",
            "CEU5500000001": "FinancialActivities",
            "CEU6500000001": "EducationAndHealthServices",
            "CEU7000000001": "LeisureAndHospitality",
            "CEU1000000001": "MiningAndLogging",
        },
    },
    "ppi": {
        "schedule_name": "Producer Price Index",
        "release_time": "08:30",
        "series": {
            "WPUFD4": "FinalDemand",
            "WPUFD49104": "CorePpi",
            "WPUFD49116": "FinalDemandLessFoodEnergyTrade",
            "WPUFD41": "FinalDemandGoods",
            "WPUFD42": "FinalDemandServices",
            "WPUFD43": "FinalDemandConstruction",
            "WPU00000000": "AllCommodities",
            "WPU01": "FarmProducts",
            "WPU02": "ProcessedFoodsAndFeeds",
            "WPU0571": "CrudePetroleum",
            "WPUFD49112": "FinalDemandGoodsLessFoods",
        },
    },
    "jolts": {
        "schedule_name": "Job Openings and Labor Turnover Survey",
        "release_time": "10:00",
        "series": {
            "JTU000000000000000JOL": "JobOpenings",
            "JTU000000000000000JOR": "JobOpeningsRate",
            "JTU000000000000000HIL": "Hires",
            "JTU000000000000000HIR": "HiresRate",
            "JTU000000000000000QUL": "Quits",
            "JTU000000000000000QUR": "QuitsRate",
            "JTU000000000000000TSL": "TotalSeparations",
            "JTU000000000000000LDL": "LayoffsAndDischarges",
        },
    },
}

print(f"Total series: {sum(len(s['series']) for s in SURVEYS.values())}")
for key, survey in SURVEYS.items():
    print(f"  {key}: {len(survey['series'])} series")

## Release Date Scraping

Scrapes BLS schedule pages (2000-2026) for exact release dates.
Caches results to `release_dates_cache.json` to avoid re-scraping.

In [ ]:
def parse_reference_period(desc_text):
    """
    Extract (year, month) from description text like
    'Consumer Price Index for December 2023' or
    'Consumer Price Index, December 1999'.
    Returns (year, month) or None.
    """
    months = {
        "january": 1, "february": 2, "march": 3, "april": 4,
        "may": 5, "june": 6, "july": 7, "august": 8,
        "september": 9, "october": 10, "november": 11, "december": 12,
    }
    m = re.search(r'(?:for|,)\s+(\w+)\s+(\d{4})', desc_text, re.IGNORECASE)
    if m:
        month_name = m.group(1).lower()
        year = int(m.group(2))
        if month_name in months:
            return (year, months[month_name])
    return None


HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}

MONTH_ABBREVS = {
    "jan": 1, "feb": 2, "march": 3, "mar": 3, "april": 4, "apr": 4,
    "may": 5, "june": 6, "jun": 6, "july": 7, "jul": 7,
    "aug": 8, "sept": 9, "sep": 9, "oct": 10, "nov": 11, "dec": 12,
}

MONTH_NAMES = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12,
}


def scrape_schedule_year_old_format(year, html):
    """
    Parse old-format BLS schedule pages (2000-2007).
    """
    releases = {}

    report_patterns = {
        "Consumer Price Index": "Consumer Price Index",
        "Consumer Price Indexes": "Consumer Price Index",
        "Producer Price Index": "Producer Price Index",
        "Producer Price Indexes": "Producer Price Index",
        "Employment Situation": "Employment Situation",
        "Job Openings": "Job Openings and Labor Turnover Survey",
    }

    text = re.sub(r'<[^>]+>', '\n', html)
    current_release_year = year

    for line in text.split('\n'):
        line = line.strip()
        if not line:
            continue

        matched_report = None
        for pattern, canonical in report_patterns.items():
            if pattern in line:
                matched_report = canonical
                break

        if not matched_report:
            if "Employment Situation" in line:
                matched_report = "Employment Situation"

        if not matched_report:
            continue

        ref = parse_reference_period(line)
        if not ref:
            continue
        ref_year, ref_month = ref

        ref_text_match = re.search(r',\s+\w+\s+\d{4}', line)
        if not ref_text_match:
            continue
        after_ref = line[ref_text_match.end():]

        date_match = re.search(
            r'(\w+)\.?\s+(\d{1,2})(?:\s*,\s*(\d{4}))?',
            after_ref
        )
        if not date_match:
            continue

        rel_month_str = date_match.group(1).lower().rstrip('.')
        rel_day = int(date_match.group(2))
        rel_year_str = date_match.group(3)

        if rel_year_str:
            rel_year = int(rel_year_str)
            current_release_year = rel_year
        else:
            rel_year = current_release_year

        rel_month = MONTH_ABBREVS.get(rel_month_str) or MONTH_NAMES.get(rel_month_str)
        if not rel_month:
            continue

        release_date = datetime(rel_year, rel_month, rel_day)
        releases[(matched_report, ref_year, ref_month)] = release_date

    return releases


def scrape_schedule_year_new_format(year, html):
    """
    Parse new-format BLS schedule pages (2008+).
    """
    releases = {}

    row_pattern = re.compile(
        r'<tr[^>]*class="release-list-(?:odd|even)-row"[^>]*>'
        r'.*?<td[^>]*class="date-cell"[^>]*><p>([^<]+)</p></td>'
        r'.*?<td[^>]*class="time-cell"[^>]*><p>([^<]*)</p></td>'
        r'.*?<td[^>]*class="desc-cell"[^>]*><p>(.*?)</p></td>',
        re.DOTALL
    )

    for match in row_pattern.finditer(html):
        date_str = match.group(1).strip()
        time_str = match.group(2).strip()
        desc_html = match.group(3).strip()

        if not time_str or time_str == '&nbsp;' or time_str == '\xa0':
            continue

        strong_match = re.search(r'<strong>([^<]+)</strong>', desc_html)
        if not strong_match:
            continue
        report_name = strong_match.group(1).strip()

        desc_text = re.sub(r'<[^>]+>', '', desc_html)

        try:
            release_date = datetime.strptime(date_str, "%A, %B %d, %Y")
        except ValueError:
            continue

        ref = parse_reference_period(desc_text)
        if ref:
            ref_year, ref_month = ref
            releases[(report_name, ref_year, ref_month)] = release_date

    return releases


def scrape_schedule_year(year):
    """
    Scrape BLS release schedule for a given year.
    """
    url = f"https://www.bls.gov/schedule/{year}/home.htm"
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"  WARNING: Could not fetch schedule for {year}: {e}")
        return {}

    html = resp.text

    if "release-list-odd-row" in html:
        return scrape_schedule_year_new_format(year, html)
    else:
        return scrape_schedule_year_old_format(year, html)


def build_release_date_map(start_year=2000, end_year=2026):
    """
    Build a mapping from (survey_key, ref_year, ref_month) to release_date.
    Caches to release_dates_cache.json to avoid re-scraping.
    """
    cache_path = Path("release_dates_cache.json")
    if cache_path.exists():
        print("Loading release dates from cache...")
        with open(cache_path) as f:
            cached = json.load(f)
        release_map = {}
        for key_str, date_str in cached.items():
            parts = key_str.split("|")
            survey_key = parts[0]
            ref_year = int(parts[1])
            ref_month = int(parts[2])
            release_map[(survey_key, ref_year, ref_month)] = datetime.strptime(date_str, "%Y-%m-%d %H:%M")
        print(f"  Loaded {len(release_map)} cached release dates")
        return release_map

    release_map = {}

    for year in range(start_year, end_year + 1):
        print(f"  Scraping schedule for {year}...")
        releases = scrape_schedule_year(year)
        count = 0

        for (report_name, ref_year, ref_month), release_date in releases.items():
            for survey_key, survey_info in SURVEYS.items():
                if report_name == survey_info["schedule_name"]:
                    release_time = survey_info["release_time"]
                    hour, minute = map(int, release_time.split(":"))
                    release_dt = release_date.replace(hour=hour, minute=minute)
                    release_map[(survey_key, ref_year, ref_month)] = release_dt
                    count += 1

        print(f"    Found {count} matching releases for our surveys")
        time.sleep(1.5)  # Respect BLS rate limits

    # Cache results
    cache_data = {}
    for (survey_key, ref_year, ref_month), release_dt in release_map.items():
        key_str = f"{survey_key}|{ref_year}|{ref_month}"
        cache_data[key_str] = release_dt.strftime("%Y-%m-%d %H:%M")

    with open(cache_path, "w") as f:
        json.dump(cache_data, f, indent=2)
    print(f"  Cached {len(release_map)} release dates to {cache_path}")

    return release_map

## BLS API Fetching

In [ ]:
def fetch_bls_data(series_ids, start_year, end_year, max_retries=3):
    """
    Fetch data from BLS API v2 for multiple series.
    Returns dict: {series_id: [{year, period, value}, ...]}
    """
    payload = {
        "seriesid": series_ids,
        "startyear": str(start_year),
        "endyear": str(end_year),
        "registrationkey": API_KEY,
        "catalog": False,
        "calculations": False,
        "annualaverage": False,
    }

    for attempt in range(max_retries):
        try:
            resp = requests.post(BLS_API_URL, json=payload, timeout=120)
            resp.raise_for_status()
            data = resp.json()

            if data.get("status") != "REQUEST_SUCCEEDED":
                msgs = data.get("message", [])
                print(f"  API warning: {msgs}")
                if "threshold" in str(msgs).lower():
                    print("  Daily API limit reached. Stopping.")
                    return {}

            result = {}
            for series in data.get("Results", {}).get("series", []):
                sid = series["seriesID"]
                points = []
                for d in series.get("data", []):
                    period = d.get("period", "")
                    if not period.startswith("M") or period == "M13":
                        continue  # Skip annual averages and non-monthly
                    points.append({
                        "year": int(d["year"]),
                        "month": int(period[1:]),
                        "value": d["value"],
                    })
                result[sid] = points
            return result

        except Exception as e:
            wait = 15 * (attempt + 1)
            print(f"  API error (attempt {attempt+1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                print(f"  Retrying in {wait}s...")
                time.sleep(wait)

    print("  All retries failed.")
    return {}


def fetch_all_survey_data(survey_key, survey_info, start_year=2000, end_year=2026):
    """
    Fetch all data for a survey, paginating in 10-year windows.
    The BLS API silently truncates responses beyond 10 years even though
    the documented limit is 20 years for registered users.
    Returns dict: {series_id: [{year, month, value}, ...]}
    """
    series_ids = list(survey_info["series"].keys())
    all_data = {sid: [] for sid in series_ids}

    current_start = start_year
    while current_start <= end_year:
        current_end = min(current_start + 9, end_year)
        print(f"  Fetching {survey_key} data: {current_start}-{current_end}...")

        result = fetch_bls_data(series_ids, current_start, current_end)

        for sid, points in result.items():
            all_data[sid].extend(points)

        current_start = current_end + 1
        if current_start <= end_year:
            time.sleep(2)  # Pause between API calls

    return all_data

## Data Assembly

In [ ]:
def assemble_survey_csv(survey_key, survey_info, api_data, release_map):
    """
    Assemble a wide-format DataFrame for a survey.
    CSV columns: time,endtime,series1,series2,...
    Only includes data points that have exact release dates.
    """
    series_map = survey_info["series"]  # {series_id: property_name}
    property_names = list(series_map.values())

    # Build a dict: (year, month) -> {property_name: value}
    period_data = {}
    for series_id, property_name in series_map.items():
        for point in api_data.get(series_id, []):
            key = (point["year"], point["month"])
            if key not in period_data:
                period_data[key] = {}
            val = point["value"]
            if val == "-":
                val = ""
            period_data[key][property_name] = val

    # Build rows, only including periods with known release dates
    rows = []
    skipped = 0
    for (year, month) in sorted(period_data.keys()):
        release_dt = release_map.get((survey_key, year, month))
        if release_dt is None:
            skipped += 1
            continue

        time_str = f"{year}{month:02d}01"  # 1st of the observation month
        endtime_str = release_dt.strftime("%Y%m%d %H:%M")

        row = {"time": time_str, "endtime": endtime_str}
        values = period_data[(year, month)]
        for prop in property_names:
            row[prop] = values.get(prop, "")
        rows.append(row)

    if skipped > 0:
        print(f"  {survey_key}: Skipped {skipped} periods without release dates")

    columns = ["time", "endtime"] + property_names
    df = pd.DataFrame(rows, columns=columns)
    return df

## Merge & Save

In [ ]:
def existing_data_path(survey_key):
    return os.path.join(LEAN_DATA_FOLDER, RELATIVE_DATA_PATH, f"{survey_key}.csv")


def temp_output_path(survey_key):
    return os.path.join(TEMP_OUTPUT_DIR, RELATIVE_DATA_PATH, f"{survey_key}.csv")


def read_existing_data(survey_key):
    path = existing_data_path(survey_key)
    if not os.path.exists(path):
        return pd.DataFrame()
    return pd.read_csv(path, header=None, dtype=str)


def merge_and_save(old_df, new_df, survey_key):
    """
    Merge new data with existing, deduplicate by time column, sort, and save.
    """
    if old_df.empty and new_df.empty:
        return pd.DataFrame()

    if old_df.empty:
        merged = new_df
    elif new_df.empty:
        merged = old_df
    else:
        # Ensure same column count
        new_df.columns = range(len(new_df.columns))
        if len(old_df.columns) != len(new_df.columns):
            # Column count mismatch - rebuild old data
            merged = new_df
        else:
            merged = pd.concat([old_df, new_df], ignore_index=True)

    merged.drop_duplicates(subset=[0], keep="last", inplace=True)
    merged.sort_values(0, inplace=True)
    merged.reset_index(drop=True, inplace=True)

    out_path = temp_output_path(survey_key)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    # Write to temp file then move atomically
    tmp_path = out_path + ".tmp"
    merged.to_csv(tmp_path, index=False, header=False)
    if os.path.exists(out_path):
        os.remove(out_path)
    os.rename(tmp_path, out_path)

    return merged

## Process

In [ ]:
full_start = time.time()

# Step 1: Build release date map
print("=" * 60)
print("Step 1: Building release date map...")
print("=" * 60)
release_map = build_release_date_map(start_year=2000, end_year=2026)
print(f"Total release dates: {len(release_map)}")

# Wait after scraping before hitting the API
print("Waiting 5s before API calls...")
time.sleep(5)

# Step 2: Process each survey
print("\n" + "=" * 60)
print("Step 2: Fetching and processing survey data...")
print("=" * 60)

for survey_key, survey_info in SURVEYS.items():
    survey_start = time.time()
    print(f"\nProcessing {survey_key.upper()}...")

    # Read existing data
    old_df = read_existing_data(survey_key)
    print(f"  Existing data: {len(old_df)} rows")

    if INCREMENTAL_MODE:
        # In incremental mode, only fetch the deployment year
        deploy_date = datetime.strptime(deployment_date_str, "%Y%m%d")
        fetch_year = deploy_date.year
        api_data = fetch_all_survey_data(survey_key, survey_info,
                                          start_year=fetch_year, end_year=fetch_year)
    else:
        # Full history mode
        api_data = fetch_all_survey_data(survey_key, survey_info,
                                          start_year=2000, end_year=2026)

    # Assemble wide-format DataFrame
    new_df = assemble_survey_csv(survey_key, survey_info, api_data, release_map)
    print(f"  New data assembled: {len(new_df)} rows")

    if new_df.empty:
        print(f"  No new data for {survey_key}")
        continue

    # Convert new_df to headerless format for merging
    new_headerless = new_df.copy()
    new_headerless.columns = range(len(new_headerless.columns))

    # Merge and save
    merged = merge_and_save(old_df, new_headerless, survey_key)

    survey_elapsed = time.time() - survey_start
    print(f"  Saved {len(merged)} rows to {temp_output_path(survey_key)} ({survey_elapsed:.1f}s)")

    # Pause between surveys to respect API limits
    time.sleep(2)

elapsed = time.time() - full_start
print(f"\n{'=' * 60}")
print(f"Processing completed in {elapsed:.1f}s")
print(f"{'=' * 60}")